# 🔑 Solutions: Lesson 2: Data Cleaning II
**Try the problems yourself first!** Looking at a solution before struggling is like watching someone else do push-ups.
Every solution below is self-contained and has a short explanation of *why* it works.

In [1]:
import sys
sys.path.append("../utils")            # lets us import course_utils.py and agri_prep.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from course_utils import *             # load_raw_farm, profile, setup_plots, ...

setup_plots()
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
np.random.seed(42)


import re, difflib
raw = load_raw_farm()

---
### Problem 2.1 &nbsp; ⭐ &nbsp; Standardise soil types

In the raw data, `soil_type` has spellings such as `'loam'`, `'LOAM'`, `'Loamy'`, `'Sand'`, `'Silty'`. 
Standardise it to exactly the five valid labels: **Loam, Clay, Sandy, Silt, Volcanic** (missing values stay missing). Print `value_counts(dropna=False)` before and after.

In [2]:
s = raw["soil_type"]
print("BEFORE:\n", s.value_counts(dropna=False).to_string())

valid = ["Loam", "Clay", "Sandy", "Silt", "Volcanic"]
lookup = {v.lower(): v for v in valid}
lookup.update({"loamy": "Loam", "sand": "Sandy", "silty": "Silt"})     # known variants

clean = s.str.strip().str.lower().map(lookup)

print("\nAFTER:\n", clean.value_counts(dropna=False).to_string())
print("\nunmatched (non-missing -> NaN):", (clean.isna() & s.notna()).sum())

BEFORE:
 soil_type
Loam        622
Volcanic    499
Clay        426
Silt        296
Sandy       278
NaN          75
CLAY         38
Loamy        37
volcanic     34
loam         27
LOAM         27
clay         27
VOLCANIC     23
sandy        22
silt         21
Silty        17
Sand         16

AFTER:
 soil_type
Loam        713
Volcanic    556
Clay        491
Silt        334
Sandy       316
NaN          75

unmatched (non-missing -> NaN): 0


**💬 Why this works:** Normalise first (strip, lower-case), then map through a dictionary. Mapping unknown values to NaN, and *counting* them, is safer than leaving them as quiet new categories.

---
### Problem 2.2 &nbsp; ⭐⭐ &nbsp; Turn 'Yes / Y / TRUE / 1' into 1 and 0

`irrigation` has values like `Yes, yes, Y, TRUE, 1, No, no, N, FALSE, 0`. Convert it to a clean 0/1 integer column. Verify nothing was left unmapped, and report the percentage of irrigated records.

In [3]:
YES, NO = {"yes", "y", "true", "1"}, {"no", "n", "false", "0"}
k = raw["irrigation"].astype(str).str.strip().str.lower()
irrigation = k.map(lambda v: 1 if v in YES else 0 if v in NO else np.nan)

print("unmapped:", irrigation.isna().sum())
print(f"irrigated: {irrigation.mean() * 100:.1f}% of records")
irrigation.astype("Int64").value_counts()

unmapped: 0
irrigated: 24.7% of records


irrigation
0    1870
1     615
Name: count, dtype: Int64

**💬 Why this works:** Boolean-like columns come in many costumes. Define the 'true' and 'false' vocabularies explicitly so that anything unexpected becomes NaN and gets noticed rather than silently becoming 0.

---
### Problem 2.3 &nbsp; ⭐⭐ &nbsp; Write a validity rule table

Define plausible bounds for `rainfall_mm` (50–3000), `avg_temp_c` (−5 to 40) and `soil_ph` (3–10). 

1. Report how many raw values violate each rule and show a few examples.
2. Which of these violations look like **sentinel codes** (-999, 99, ...)?

In [4]:
rules = {"rainfall_mm": (50, 3000), "avg_temp_c": (-5, 40), "soil_ph": (3, 10)}
for col, (lo, hi) in rules.items():
    s = raw[col]
    bad = (s < lo) | (s > hi)
    print(f"{col:12s} allowed [{lo}, {hi}] -> {bad.sum():3d} violations, values: {sorted(s[bad].unique().tolist())[:6]}")

rainfall_mm  allowed [50, 3000] ->   7 violations, values: [-317.0]
avg_temp_c   allowed [-5, 40] ->   5 violations, values: [-999.0, 99.0]
soil_ph      allowed [3, 10] ->  10 violations, values: [-1.0, 0.0, 14.5, 99.0]


**💬 Why this works:** A rule table turns vague suspicion into a measurable report. Values like -999 and 99 are classic 'sentinel' codes: someone's way of writing 'unknown' inside a numeric column.

---
### Problem 2.4 &nbsp; ⭐⭐ &nbsp; Three outlier detectors, one column

For `nitrogen_ppm` (use only values > 0, drop NaN), count the outliers flagged by:
- **Z-score** (|z| > 3),
- **IQR / Tukey fences** (1.5 × IQR),
- **Modified z-score using the median and MAD** (> 3.5).

Which method flags the most? Which one is the *least robust* and why?

In [5]:
x = raw["nitrogen_ppm"]
x = x[x > 0].dropna()

z = (x - x.mean()) / x.std()
q1, q3 = x.quantile([0.25, 0.75]); iqr = q3 - q1
mad = (x - x.median()).abs().median()
mz = 0.6745 * (x - x.median()) / mad

counts = {
    "z-score |z|>3":      int((z.abs() > 3).sum()),
    "IQR 1.5x":           int(((x < q1 - 1.5 * iqr) | (x > q3 + 1.5 * iqr)).sum()),
    "modified z (MAD)>3.5": int((mz.abs() > 3.5).sum()),
}
print(pd.Series(counts))

z-score |z|>3            9
IQR 1.5x                11
modified z (MAD)>3.5     9
dtype: int64


**💬 Why this works:** The z-score uses the mean and standard deviation (numbers the outliers themselves inflate) so it can 'hide' (mask) extreme points. Median/MAD and quartiles are **robust**: a few extremes barely move them.

---
### Problem 2.5 &nbsp; ⭐⭐⭐ &nbsp; Outlier relative to what?

Load the lesson-clean data with `load_lesson_clean()` (finish notebook 02 first: it creates this file).

1. Flag `yield_t_ha` outliers with the IQR rule **globally**, and again **within each crop**.
2. Which crop produces most of the *global* flags? Is that crop actually behaving strangely?
3. Write one sentence on what this teaches about outlier detection.

In [6]:
df = load_lesson_clean()

def iqr_mask(s, k=1.5):
    q1, q3 = s.quantile([0.25, 0.75]); i = q3 - q1
    return (s < q1 - k * i) | (s > q3 + k * i)

global_flag  = iqr_mask(df["yield_t_ha"])
by_crop_flag = df.groupby("crop")["yield_t_ha"].transform(iqr_mask).astype(bool)

print("flagged globally :", global_flag.sum())
print("flagged per crop :", by_crop_flag.sum())
print("\nGlobal flags by crop:\n", df.loc[global_flag, "crop"].value_counts().to_string())
print("\nPer-crop flags by crop:\n", df.loc[by_crop_flag, "crop"].value_counts().to_string())

flagged globally : 191
flagged per crop : 34

Global flags by crop:
 crop
Potato    190
Maize       1

Per-crop flags by crop:
 crop
Sorghum    23
Beans       4
Maize       3
Potato      2
Wheat       2


**💬 Why this works:** Potatoes naturally yield 3–10× more tonnes per hectare than other crops, so a *global* rule brands healthy potato farms as 'outliers'. An outlier is only strange **relative to its peer group**: detect within groups when the groups differ.